# Session 7. Observability of agent systems

**The agent answers. Nobody can say what it cost.**

- today: traces end to end, cost per node, prompts with versions, the first test
- one patient all session: a pre-built trip planner that wastes tokens
- next session it deploys; today we learn to see it

## Why observability before deploy

**`print` stopped paying for itself at three nodes. Session 8 buries it.**

- behind a server, nobody reads a notebook's stdout
- what you cannot see, you fix by guessing, and guessing bills the endpoint
- traces are also defense evidence: proof your agent really ran

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## The patient

**A pre-built weekend trip planner, deliberately wasteful.**

- read it as an artifact: today we read traces, not write agents
- one hardcoded sights catalogue, a plan node, a polish node
- the bug is invisible in the code and obvious in the trace

In [ ]:
from langchain_core.tools import tool

# a fixed catalogue: no network, same answer every run
SIGHTS = {
    "lisbon": "Belem Tower; Tram 28 through Alfama; the miradouro viewpoints; LX Factory",
    "porto": "Livraria Lello; the Ribeira riverfront; port wine cellars; Dom Luis I bridge",
    "seville": "Real Alcazar; the cathedral and La Giralda; Plaza de Espana; Triana",
    "valencia": "City of Arts and Sciences; the Turia gardens; the Central Market",
}


@tool  # the docstring is the description the model reads
def city_sights(city: str) -> str:
    """Look up the must-see sights of a city for a weekend trip."""
    key = city.strip().lower()
    if key not in SIGHTS:
        # a miss the model can act on, not an exception
        return f"No entry for {city!r}. Known cities: {', '.join(sorted(SIGHTS))}."
    return SIGHTS[key]


print(city_sights.invoke({"city": "Lisbon"}))
print(city_sights.invoke({"city": "Madrid"}))  # the miss lists what would work

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

PLAN_PROMPT = (
    "You plan weekend city trips. Look up sights with the tool, "
    "then draft a one-paragraph itinerary."
)
POLISH_PROMPT = "Rewrite the draft itinerary in a warm, friendly tone. Keep every fact."

strong = chat_model("strong")  # one object; watch both nodes below use it


def make_plan(model):  # the model is an argument: remember this seam
    bound = model.bind_tools([city_sights])

    def plan(state: MessagesState) -> dict:
        messages = [{"role": "system", "content": PLAN_PROMPT}, *state["messages"]]
        return {"messages": [bound.invoke(messages)]}

    return plan


def make_polish(model):
    def polish(state: MessagesState) -> dict:
        messages = [{"role": "system", "content": POLISH_PROMPT}, *state["messages"]]
        return {"messages": [model.invoke(messages)]}

    return polish


def route(state: MessagesState) -> str:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "polish"  # every finished draft takes a second trip


builder = StateGraph(MessagesState)
builder.add_node("plan", make_plan(strong))
builder.add_node("tools", ToolNode([city_sights]))
builder.add_node("polish", make_polish(strong))  # the same strong object again
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", route, ["tools", "polish"])
builder.add_edge("tools", "plan")
builder.add_edge("polish", END)
wasteful = builder.compile()

print(wasteful.get_graph().draw_mermaid())

In [ ]:
TRIP_QUESTION = "Plan me a weekend in Lisbon, one paragraph."

result = wasteful.invoke(
    {"messages": [{"role": "user", "content": TRIP_QUESTION}]},
    config={"recursion_limit": 8},  # explicit always: a runaway loop bills the endpoint
)

print(len(result["messages"]), "messages")
print(result["messages"][-1].content)

**Five messages, one answer, zero insight.**

- which node was slow? what did polish change? what did each call cost?
- the model ran three times; `print` showed one string
- we need a second reader that sees every step, not more prints

## Langfuse

**A server you own, an SDK, a UI.**

- one trace per run, a span per node, a generation per model call
- course default: the session-2 compose stack, local, no limits; it joins the session-8 deploy template
- cloud free tier exists: email signup, 50k units a month, 2 users, 30-day retention (numbers as of August)

In [ ]:
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

# LANGFUSE_BASE_URL is the current name; LANGFUSE_HOST still honored
langfuse_client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", langfuse_client.auth_check())

langfuse_handler = CallbackHandler()  # no kwargs: metadata carries session and user later

In [ ]:
traced = wasteful.invoke(
    {"messages": [{"role": "user", "content": TRIP_QUESTION}]},
    config={"recursion_limit": 8, "callbacks": [langfuse_handler]},
)
langfuse_client.flush()  # kernels never exit; nothing sends without this

print(traced["messages"][-1].content)

**The handler rides in the config. The graph never changed.**

- the same graph runs traced or untraced; nothing inside it knows
- the constructor takes no session or user kwargs: metadata does that, next
- one handler serves every run; each invoke still becomes its own trace, metadata does the grouping
- `flush()` is a rule: a notebook kernel never exits, so nothing sends without it

## Reading the trace

**The trace is your graph, executed.**

- aggregated view: repeated nodes merged, a counter on the plan-tools loop
- expanded view: the loop unrolled, one span per visit
- before scrolling on: how many generations did the strong model make?

**Three generations, and the last re-reads everything the second wrote.**

- polish sent the entire finished draft back to the same expensive model
- the latency column names the slowest span; token usage sits on each generation
- the cost view attributes spend per node; unpriced model? tokens tell the same story
- the code never looked wrong. The trace does

In [ ]:
run_config = {
    "recursion_limit": 8,
    "callbacks": [langfuse_handler],
    "metadata": {
        "langfuse_session_id": "trip-dialogue-01",  # one id per dialogue
        "langfuse_user_id": "demo-student",  # your GitHub handle in the practice
        "langfuse_tags": ["wasteful"],  # a list; filter runs by tag in the UI
    },
}

turn_one = wasteful.invoke(
    {"messages": [{"role": "user", "content": TRIP_QUESTION}]}, config=run_config
)
followup = {"role": "user", "content": "Now make it work for a rainy day."}
turn_two = wasteful.invoke(
    {"messages": [*turn_one["messages"], followup]}, config=run_config
)
langfuse_client.flush()

print("two traces, one session:", run_config["metadata"]["langfuse_session_id"])

**Three metadata keys, three questions answered.**

- `langfuse_session_id`: turns grouped into one dialogue, in order
- `langfuse_user_id`: cost per person, the number a budget owner asks for
- `langfuse_tags`: filterable experiments; tag before you compare anything

## The fix, read from the trace

**Merge polish into plan: one prompt asks for the final tone.**

- the trace named the guilty node; a code read-through never would have
- if a separate rewrite step must stay, demote it instead:

```python
builder.add_node("polish", make_polish(chat_model("cheap")))  # the cheap restyle
```

In [ ]:
# polish's whole job becomes one sentence of plan's prompt
MERGED_PROMPT = PLAN_PROMPT + " Write it in a warm, friendly tone."


def make_merged(model):
    bound = model.bind_tools([city_sights])

    def plan(state: MessagesState) -> dict:
        messages = [{"role": "system", "content": MERGED_PROMPT}, *state["messages"]]
        return {"messages": [bound.invoke(messages)]}

    return plan


def route_merged(state: MessagesState) -> str:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END  # no second trip: the draft is the answer


builder = StateGraph(MessagesState)
builder.add_node("plan", make_merged(strong))  # the same strong object, one node fewer
builder.add_node("tools", ToolNode([city_sights]))
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", route_merged, ["tools", END])
builder.add_edge("tools", "plan")
merged = builder.compile()

fixed = merged.invoke(
    {"messages": [{"role": "user", "content": TRIP_QUESTION}]},
    config={"recursion_limit": 8},
)
print(len(fixed["messages"]), "messages, was", len(result["messages"]))
print(fixed["messages"][-1].content)

In [ ]:
merged_config = {
    "recursion_limit": 8,
    "callbacks": [langfuse_handler],  # same handler as before; only the tag differs
    "metadata": {"langfuse_tags": ["merged"]},  # the before/after pair filters by tag
}

retraced = merged.invoke(
    {"messages": [{"role": "user", "content": TRIP_QUESTION}]}, config=merged_config
)
langfuse_client.flush()
print("open one wasteful trace and the merged trace side by side; count generations")

## Prompts with versions

**A prompt in code needs a redeploy. A prompt in Langfuse needs a save.**

- `create_prompt` stores it; a label like `production` marks the serving version
- the playground opens a trace's prompt, edits it, saves a new version
- config metadata key `langfuse_prompt` ties generations to the version they used

In [ ]:
stored = langfuse_client.create_prompt(
    name="trip-planner-plan",
    prompt=MERGED_PROMPT,  # the string leaves the code and gains versions
    labels=["production"],  # the label a fetch asks for
)
print("stored:", stored.name, "| version:", stored.version, "| label production")

In [ ]:
plan_prompt = langfuse_client.get_prompt("trip-planner-plan", label="production")
print("fetched:", plan_prompt.name, "| version:", plan_prompt.version)

MERGED_PROMPT = plan_prompt.compile()  # plan reads this global at call time
print(MERGED_PROMPT)

refetched = merged.invoke(  # no rebuild, no restart: the next run serves the fetch
    {"messages": [{"role": "user", "content": TRIP_QUESTION}]},
    config={"recursion_limit": 8, "callbacks": [langfuse_handler]},
)
langfuse_client.flush()
print(refetched["messages"][-1].content)

## Why not LangSmith

**Same category of tool, different ownership.**

- LangSmith: hosted service, closed code, external limits you do not set
- Langfuse: MIT code, your compose file, your retention rules
- Studio's graph view -> the agent-graph trace here; `evaluate()` -> `run_experiment`, session 11

## The first test

**The double call was found by eyeball. Eyes do not scale.**

- the next regression should be caught by a robot: no tokens, no key, no server
- `FakeListChatModel` replays canned replies, in order
- remember the seam: any model-shaped argument fits `make_polish`

In [ ]:
from langchain_core.language_models import FakeListChatModel

fake = FakeListChatModel(responses=["first canned reply", "second canned reply"])
print(fake.invoke("anything").content)
print(fake.invoke("anything, again").content)  # replies come back in order

try:  # the reason today's test targets a tool-free node
    fake.bind_tools([city_sights])
except NotImplementedError:
    print("bind_tools: NotImplementedError, with an empty message")

In [ ]:
from langchain_core.messages import HumanMessage

polish_node = make_polish(FakeListChatModel(responses=["A warm, walkable weekend."]))
update = polish_node({"messages": [HumanMessage("Plan me a weekend in Lisbon.")]})

assert list(update) == ["messages"]  # a node returns a partial update
assert update["messages"][-1].content == "A warm, walkable weekend."
print("passed: no key read, no request sent, nothing billed")

**Green, and nothing was mocked at the transport layer.**

- the node never noticed: the model is an argument, so a fake slid in
- an assert in a notebook dies with the kernel; a file under `tests/` survives
- the file below is self-contained; in your repo, import your real factory

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

TEST_FILE = '''
from langchain_core.language_models import FakeListChatModel
from langchain_core.messages import HumanMessage


def make_polish(model):
    def polish(state):
        return {"messages": [model.invoke(state["messages"])]}

    return polish


def test_polish_returns_one_ai_message():
    node = make_polish(FakeListChatModel(responses=["A warm, walkable weekend."]))
    update = node({"messages": [HumanMessage("Plan me a weekend in Lisbon.")]})
    assert list(update) == ["messages"]
    assert update["messages"][-1].content == "A warm, walkable weekend."
'''

test_dir = Path(tempfile.mkdtemp())  # in your repo this is tests/, committed
(test_dir / "test_polish.py").write_text(TEST_FILE)

report = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_dir), "-q"],
    capture_output=True,
    text=True,
)
print(report.stdout.strip().splitlines()[-1])  # this line goes into runs/session-07.md
assert report.returncode == 0, report.stdout  # a failed test fails the notebook too

**The test pins shape and wiring, not wording or quality.**

- proved: one AI message out, keyed under `messages`, the model actually called
- not proved: the itinerary is any good; session 11 measures that
- `bind_tools` fails on the fake, so tool-calling nodes wait for session 11

## Practice

**Instrument your own assistant or project agent, end to end.**

1. the handler goes through every `invoke` your repository makes
2. one `langfuse_session_id` per dialogue; `langfuse_user_id` is your GitHub handle; at least one tag
3. run one real multi-tool dialogue and open its trace

**Answer from data, not from memory.**

4. name your slowest node and your priciest node from the UI columns
5. cost in tokens always; for currency, add your model's prices under Settings -> Models
6. move your system prompt into prompt management, label it, fetch it by label
7. one `FakeListChatModel` test for a tool-free node, under `tests/`

**Required artifact: `runs/session-07.md`, a trace export beside it.**

- two sentences naming your slowest and priciest node, from your own data
- the prompt name and label your code now fetches
- the pytest output line, and the test file committed
- tracing is mandatory from today to the defense: traces are your proof of work

**Stretch, if you finish early.**

- tag a cheap-versus-strong run pair and read the cost difference by tag
- test your router next: a pure function needs no fake at all
- bring one trace to the projector: the session ends by reading a few together

## Today, in one card

**The trace is your graph, executed: what the code never shows, the trace names.**

**You can now defend:**
- the handler rides in the config, so the same graph runs traced or untraced and nothing inside it knows
- `langfuse_session_id`, `langfuse_user_id` and tags in the metadata group turns, attribute cost per person, and make runs comparable
- a scripted `FakeListChatModel` tests a node's shape and wiring for free; whether the answer is any good is session 11

**In your repository:** `runs/session-07.md`, your slowest and priciest node from the UI, the prompt label your code fetches, the green pytest line.
**The trap of the day:** nothing arrives in Langfuse because a notebook kernel never exits: nothing sends until `flush()`.
**Ask yourself:** a two-node graph reads fine and bills three model calls: what in the trace names the guilty node, and what is the fix?
**Next time:** deploy: the compose file grows `api` and `db`, the agent moves behind FastAPI, and the traces keep flowing.